# Degree Attainment Pipeline (Google Colab)

This notebook runs the same core process used in the local project:

1. upload a raw Excel cohort file
2. clean and standardize the data
3. build the primary analysis cohort
4. build grouped interpretive variables
5. fit a statsmodels logistic regression
6. download the processed data and summary files


In [ ]:
!pip -q install pandas numpy openpyxl statsmodels scikit-learn

In [ ]:
from pathlib import Path
import io
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from google.colab import files
from sklearn.metrics import accuracy_score, roc_auc_score

RENAME_MAP = {
    'Term_Code': 'term_code',
    'Term_Semester': 'term_semester',
    'Student ID': 'student_id',
    'Degree': 'degree',
    'StudentType': 'student_type',
    'Gender': 'gender',
    'IPEDS_Ethn': 'ipeds_ethnicity',
    'Ethn_Detail': 'ethnicity_detail',
    'age_cat': 'age_category',
    'Residency': 'residency',
    'Athlete': 'athlete',
    'Disability': 'disability',
    'FosterYouth': 'foster_youth',
    'Intl_Stdnt': 'international_student',
    'Veteran': 'veteran',
    'OCCProgram_Code': 'program_code',
    'OCCProgram_Desc': 'program_description',
    'distr': 'district_status',
    'hs_feeder': 'high_school_feeder',
    'BOG_Elig': 'bog_eligible',
    'ED_GOAL': 'education_goal',
    'AGE': 'age',
    'FT_PT': 'attendance_status',
    'ZIPCODE': 'zipcode',
    'Degree_obtained': 'degree_obtained',
}
ZERO_ONE_COLUMNS = ['athlete', 'veteran', 'degree_obtained']
YN_COLUMNS = ['disability', 'foster_youth', 'international_student']
INCLUDED_GOALS = {
    'AA Degree w/Transfer Bach.',
    'AA Degree w/out Transfer',
    "Bachelor's Degree or higher",
    'Certificate Only',
    'Two Yr. Vocational Degree',
    '4 yr college student meet 4 yr',
}
FORMULA = """
degree_obtained ~ age
    + C(attendance_status, Treatment(reference='PT'))
    + C(gender_grouped, Treatment(reference='F'))
    + C(ethnicity_grouped, Treatment(reference='Hispanic'))
    + C(residency_grouped, Treatment(reference='California Resident'))
    + athlete
    + disability
    + international_student
    + C(district_status, Treatment(reference='In-District'))
    + C(high_school_feeder, Treatment(reference='Not a Feeder HS'))
    + C(bog_eligible, Treatment(reference='No'))
    + C(education_goal_grouped, Treatment(reference='Transfer-Oriented'))
"""

def standardize_strings(df):
    for column in df.columns:
        if pd.api.types.is_object_dtype(df[column]):
            df[column] = df[column].map(lambda value: value.strip() if isinstance(value, str) else value)
    return df

def normalize_binary_columns(df):
    for column in ZERO_ONE_COLUMNS:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors='coerce').astype('Int64')
    yn_map = {'Y': 1, 'N': 0, 'Yes': 1, 'No': 0}
    for column in YN_COLUMNS:
        if column in df.columns:
            df[column] = df[column].map(yn_map).astype('Int64')
    return df

def prepare_model_dataset(file_bytes):
    df = pd.read_excel(io.BytesIO(file_bytes))
    df = df.rename(columns=RENAME_MAP)
    df = standardize_strings(df)
    df = normalize_binary_columns(df)
    df['student_id'] = df['student_id'].astype(str)
    df['term_code'] = df['term_code'].astype(str)
    return df

def build_primary_analysis_cohort(df):
    filtered = df[df['attendance_status'].isin(['FT', 'PT'])].copy()
    filtered = filtered[filtered['education_goal'].isin(INCLUDED_GOALS)].copy()
    return filtered

def group_residency(value):
    if value == 'California Resident':
        return 'California Resident'
    if value in {'Foreign', 'Non-Resident-Out of state'}:
        return 'Nonresident/Foreign'
    return 'AB540/Exempt/Other'

def group_ethnicity(value):
    if value in {'Hispanic', 'White', 'Asian', 'Black'}:
        return value
    return 'Other/Multiple/Unknown'

def group_gender(value):
    if value in {'F', 'M'}:
        return value
    return 'Other/Unknown'

def group_education_goal(value):
    if value in {'AA Degree w/Transfer Bach.', "Bachelor's Degree or higher", '4 yr college student meet 4 yr'}:
        return 'Transfer-Oriented'
    if value == 'AA Degree w/out Transfer':
        return 'Associate Degree'
    if value in {'Certificate Only', 'Two Yr. Vocational Degree'}:
        return 'Certificate/Vocational'
    return 'Other'

def build_interpretive_dataset(df):
    grouped = df.copy()
    grouped['residency_grouped'] = grouped['residency'].map(group_residency)
    grouped['ethnicity_grouped'] = grouped['ipeds_ethnicity'].map(group_ethnicity)
    grouped['gender_grouped'] = grouped['gender'].map(group_gender)
    grouped['education_goal_grouped'] = grouped['education_goal'].map(group_education_goal)
    return grouped

def fit_model(df):
    model = smf.glm(formula=FORMULA, data=df, family=sm.families.Binomial())
    result = model.fit()
    predicted_prob = result.predict(df)
    predicted_class = (predicted_prob >= 0.5).astype(int)
    accuracy = accuracy_score(df['degree_obtained'], predicted_class)
    roc_auc = roc_auc_score(df['degree_obtained'], predicted_prob)
    null_model = smf.glm('degree_obtained ~ 1', data=df, family=sm.families.Binomial()).fit()
    mcfadden_r2 = 1 - (result.llf / null_model.llf)
    conf = result.conf_int()
    coef_df = pd.DataFrame({
        'term': result.params.index,
        'coefficient': result.params.values,
        'odds_ratio': np.exp(result.params.values),
        'ci_lower_odds_ratio': np.exp(conf[0].values),
        'ci_upper_odds_ratio': np.exp(conf[1].values),
        'p_value': result.pvalues.values,
    }).sort_values('p_value')
    metrics = {'accuracy': accuracy, 'roc_auc': roc_auc, 'mcfadden_r2': mcfadden_r2, 'row_count': len(df)}
    return coef_df, metrics


In [ ]:
uploaded = files.upload()
file_name = next(iter(uploaded))
raw_bytes = uploaded[file_name]

cleaned = prepare_model_dataset(raw_bytes)
primary = build_primary_analysis_cohort(cleaned)
interpretive = build_interpretive_dataset(primary)
coef_df, metrics = fit_model(interpretive)

cleaned.to_csv('cohort_model_dataset.csv', index=False)
primary.to_csv('primary_analysis_cohort.csv', index=False)
interpretive.to_csv('interpretive_primary_analysis_cohort.csv', index=False)
coef_df.to_csv('statsmodels_degree_obtained_coefficients.csv', index=False)

print('Rows in primary cohort:', metrics['row_count'])
print('Accuracy:', round(metrics['accuracy'], 3))
print('ROC AUC:', round(metrics['roc_auc'], 3))
print('McFadden R2:', round(metrics['mcfadden_r2'], 3))

coef_df.head(15)

In [ ]:
files.download('statsmodels_degree_obtained_coefficients.csv')
files.download('primary_analysis_cohort.csv')